# Testing 11 · Prostate OncoPanel IHC outcomes (pre-existing CSVs)

This loads the two step-2 CSVs written by the original prostate_ihc scripts and summarises them:
- `ihc_extraction/results/oncopanel_prostate_specimen_ihc.csv`: one row per OncoPanel prostate specimen (match status, NE / AR-axis flags, call per marker);
- `ihc_extraction/results/oncopanel_prostate_ihc_long.csv`: one row per stain mention on the matched pathology reports.

No raw data are read and nothing is rerun or overwritten. The extraction code in `COMPASS/prostate_ihc` is not run here.
Notebook `text_interpretability/post_landmark/11_LR_for_IHC.ipynb` uses these CSVs as its outcomes.


In [ ]:
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'survival_common').is_dir()), None)
if ROOT is None: raise RuntimeError('Run from the repository or COMPASS/notebooks')
sys.path.insert(0, str(ROOT))
import polars as pl
from IPython.display import display
from COMPASS.prostate_ihc import oncopanel_ihc as op
from COMPASS.prostate_ihc import paths as ihc_paths
from COMPASS.multivariable_models import ihc_lr as il

RESULTS_DIR   = ihc_paths.default_paths().results_dir   # env: COMPASS_IHC_EXTRACTION_ROOT
SPECIMEN_PATH = RESULTS_DIR / ihc_paths.ONCOPANEL_SPECIMEN_FILE
LONG_PATH     = RESULTS_DIR / ihc_paths.ONCOPANEL_LONG_FILE
for path in (SPECIMEN_PATH, LONG_PATH):
    print(f"{path}  exists={path.exists()}")


## Step-2 CSVs: counts, phenotype mix, label agreement


In [ ]:
spec = pl.read_csv(SPECIMEN_PATH, infer_schema_length=None)
long = pl.read_csv(LONG_PATH, infer_schema_length=None)
print(f"{spec.height:,} OncoPanel prostate specimens; {long.height:,} stain rows on matched reports")
for name, table in op.summary_tables(spec).items():
    print(f"\n=== {name} ===")
    display(table.to_pandas())


## What the LR notebook sees

These are the outcome flags and dates `ihc_lr.load_oncopanel_ihc` derives from the two CSVs, and the per-marker stain frequency on matched specimens.

In [ ]:
spec_pd, long_pd = spec.to_pandas(), long.to_pandas()
lr_spec = il.load_oncopanel_ihc(spec_pd, long_pd)
display(lr_spec[["path_matched", "ihc_any", "ne_pos", "nepc_like"]].sum().rename("specimens").to_frame())
print(f"stained specimens without a pathology date: "
      f"{int((lr_spec['ihc_any'] & lr_spec['specimen_date'].isna()).sum()):,}")
display(il.marker_frequency(spec_pd))